<a href="https://colab.research.google.com/github/VIJO258/OIBSIP/blob/main/Voice_Assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
!pip install -q SpeechRecognition pydub
!apt-get -qq install ffmpeg

In [6]:
import speech_recognition as sr
import datetime
import webbrowser
import base64
import io
from pydub import AudioSegment
from IPython.display import display, Javascript, HTML
from google.colab.output import eval_js
from urllib.parse import quote

# ---------------------------------
# VOICE ASSISTANT PROJECT
# ---------------------------------

recognizer = sr.Recognizer()

# Text-to-speech using browser speaker
def speak(text):
    print("Assistant:", text)

    js = """
    (text) => {
        let speech = new SpeechSynthesisUtterance(text);
        speech.lang = 'en-US';
        window.speechSynthesis.speak(speech);
    }
    """

    from google.colab.output import eval_js
    from IPython.display import Javascript, display

    display(Javascript("""
    window.speakText = (text) => {
        let speech = new SpeechSynthesisUtterance(text);
        speech.lang = 'en-US';
        window.speechSynthesis.speak(speech);
    };
    """))

    eval_js("speakText(" + __import__('json').dumps(text) + ")")


# Record voice using browser microphone
def listen():

    display(Javascript("""
    async function recordAudio(seconds) {
        const stream = await navigator.mediaDevices.getUserMedia({audio:true});
        const recorder = new MediaRecorder(stream);
        let chunks = [];

        recorder.ondataavailable = e => chunks.push(e.data);

        recorder.start();

        await new Promise(resolve =>
            setTimeout(resolve, seconds * 1000)
        );

        recorder.stop();

        await new Promise(resolve =>
            recorder.onstop = resolve
        );

        stream.getTracks().forEach(track => track.stop());

        const blob = new Blob(chunks, {type: 'audio/webm'});
        const reader = new FileReader();

        return await new Promise(resolve => {
            reader.onloadend = () => resolve(
                reader.result.split(',')[1]
            );
            reader.readAsDataURL(blob);
        });
    }
    """))

    print("Listening... Speak now!")

    audio_data = eval_js("recordAudio(5)")

    audio_bytes = base64.b64decode(audio_data)

    # Convert browser audio into WAV format
    audio = AudioSegment.from_file(
        io.BytesIO(audio_bytes),
        format="webm"
    )

    wav_data = io.BytesIO()
    audio.export(wav_data, format="wav")
    wav_data.seek(0)

    with sr.AudioFile(wav_data) as source:
        recorded_audio = recognizer.record(source)

    try:
        text = recognizer.recognize_google(recorded_audio)
        print("You said:", text)
        return text.lower()

    except sr.UnknownValueError:
        speak("Sorry, I could not understand you.")
        return ""

    except sr.RequestError:
        speak("Speech recognition service is unavailable.")
        return ""


# Process voice commands
def voice_assistant():

    speak("Hello! I am your voice assistant.")

    while True:

        command = listen()

        if command == "":
            continue

        # Greeting
        elif "hello" in command or "hi" in command:
            speak("Hello! How can I help you?")

        # Current time
        elif "time" in command:
            current_time = datetime.datetime.now().strftime("%I:%M %p")
            speak("The current time is " + current_time)

        # Today's date
        elif "date" in command:
            current_date = datetime.datetime.now().strftime("%d %B %Y")
            speak("Today's date is " + current_date)

        # Google search
        elif "search" in command:
            topic = command.replace("search", "", 1).strip()

            if topic:
                speak("Searching Google for " + topic)

                search_url = "https://www.google.com/search?q=" + quote(topic)

                display(HTML(
                    f'<a href="{search_url}" target="_blank">'
                    f'Click here to search Google for {topic}</a>'
                ))
            else:
                speak("Please tell me what you want to search.")

        # Exit assistant
        elif "stop" in command or "exit" in command or "bye" in command:
            speak("Goodbye! Have a nice day.")
            break

        else:
            speak("Sorry, I don't know that command.")

/usr/local/lib/python3.13/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.13/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.13/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.13/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):
